# Diagnostik: dekomposisi, stasioneritas, ACF/PACF
**timeseries › diagnostics** · notebook `decompose_stationarity`

**Kapan dipakai:** Langkah pertama pada data time series baru: lihat tren/musiman, perlu diferensiasi atau tidak, dan orde ARIMA kasar.

**Jebakan umum:** Uji ADF dan KPSS bisa bertentangan; baca keduanya dan lihat grafik. Jangan menyimpulkan dari satu uji.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/timeseries/forecasting/co2_monthly.csv"; DATE_COL = "date"; VALUE_COL = "value"; FREQ = "MS"; M = 12
OUT = out_dir("timeseries", "diagnostics", "decompose_stationarity")
from superdb.timeseries import load_series
y = load_series(PATH, DATE_COL, VALUE_COL, FREQ); print(len(y), "titik")

## 2. Dekomposisi STL: tren, musiman, sisa
Kekuatan tren/musiman (0-1) mengukur seberapa dominan komponen itu terhadap sisa.

In [ ]:
from statsmodels.tsa.seasonal import STL
r = STL(y, period=M, robust=True).fit()
fig = r.plot(); fig.set_size_inches(10, 6); plt.tight_layout(); plt.savefig(OUT / "stl.png", dpi=130); plt.show()
Ft = max(0, 1 - r.resid.var() / (r.trend + r.resid).var()); Fs = max(0, 1 - r.resid.var() / (r.seasonal + r.resid).var())
print(f"kekuatan tren = {Ft:.2f} | kekuatan musiman = {Fs:.2f}")

## 3. Stasioneritas: ADF dan KPSS
- ADF: H0 = ada akar unit (tidak stasioner). p kecil → stasioner.
- KPSS: H0 = stasioner. p kecil → tidak stasioner.
Idealnya keduanya sepakat; bila bertentangan, lihat grafik dan coba diferensiasi.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
def tests(x, label):
    a = adfuller(x.dropna(), autolag="AIC")[1]; k = kpss(x.dropna(), regression="c", nlags="auto")[1]
    verdict = "stasioner" if (a < .05 and k >= .05) else ("tidak stasioner" if (a >= .05 and k < .05) else "bertentangan/ambigu")
    return {"seri": label, "p_ADF": round(a, 4), "p_KPSS": round(k, 4), "kesimpulan": verdict}
rows = [tests(y, "asli"), tests(y.diff(), "diferensi-1"), tests(y.diff(M), f"diferensi musiman ({M})"), tests(y.diff(M).diff(), "musiman + diferensi-1")]
display(pd.DataFrame(rows))

## 4. ACF/PACF untuk memperkirakan orde ARIMA
Pada seri yang sudah stasioner: ACF terpotong pada lag q → MA(q); PACF terpotong pada lag p → AR(p). Lonjakan di lag musiman (M, 2M) → komponen musiman.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
z = y.diff(M).diff().dropna()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2)); plot_acf(z, lags=3 * M, ax=ax[0]); plot_pacf(z, lags=min(3 * M, len(z) // 2 - 1), ax=ax[1], method="ywm")
plt.tight_layout(); plt.savefig(OUT / "acf_pacf.png", dpi=130); plt.show()

## 5. Apakah perlu transformasi (log / Box-Cox)?
Bila amplitudo musiman membesar seiring level, variansi tidak konstan: model multiplikatif atau log membantu.

In [ ]:
from scipy import stats
pos = y[y > 0]
lam = stats.boxcox_normmax(pos, method="mle") if len(pos) == len(y) else None
yr = y.groupby(y.index.year).agg(["mean", "std"]); corr = yr["mean"].corr(yr["std"])
print(f"korelasi (rata-rata tahunan, std tahunan) = {corr:.2f} -> {'variansi naik dengan level: pertimbangkan log/multiplikatif' if corr > .6 else 'variansi relatif konstan'}")
print("lambda Box-Cox (MLE):", None if lam is None else round(float(lam), 3), "(≈0 log, ≈1 tanpa transformasi)")
save_json({"trend_strength": float(Ft), "seasonal_strength": float(Fs), "tests": rows, "var_level_corr": float(corr), "boxcox_lambda": None if lam is None else float(lam)}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Tren kuat + tak stasioner → diferensiasi (d=1) atau model dengan tren. Musiman kuat → D=1 atau Holt-Winters/STL.
- Lanjut ke `notebooks/timeseries/forecasting/` dan bandingkan dengan baseline naive musiman.